# Part D: Cross-Validation Strategies


## Tasks Covered
- K-Fold Cross Validation
- Stratified K-Fold Cross Validation
- Leave-One-Out Cross Validation (LOOCV)
- Time Series Split
- Performance Comparison


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    LeaveOneOut,
    TimeSeriesSplit,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

df = pd.read_csv('Advanced Regression HousePrice.csv')
df.head()


,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


In [2]:
if 'sale_date' in df.columns:
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df = df.sort_values('sale_date')
    df['sale_year'] = df['sale_date'].dt.year
    df['sale_month'] = df['sale_date'].dt.month

target_col = 'house_price_inr'

X = df.drop(columns=[target_col])

if 'sale_date' in X.columns:
    X = X.drop(columns=['sale_date'])

y = df[target_col]

print(X.shape, y.shape)


(3800, 12) (3800,)


In [3]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])


In [4]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

kfold_scores = cross_val_score(
    model,
    X,
    y,
    cv=kfold,
    scoring='r2'
)

print("KFold Scores")
print(kfold_scores)
print("Mean:", kfold_scores.mean())


KFold Scores
[0.91529204 0.91770729 0.9240218  0.9070823  0.92229166]
Mean: 0.9172790203718444


In [5]:
y_bins = pd.qcut(
    y,
    q=5,
    labels=False,
    duplicates='drop'
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

strat_scores = cross_val_score(
    model,
    X,
    y,
    cv=skf.split(X, y_bins),
    scoring='r2'
)

print("Stratified Scores")
print(strat_scores)
print("Mean:", strat_scores.mean())


Stratified Scores
[0.9205613  0.90939372 0.9205026  0.91933788 0.91891468]
Mean: 0.9177420341302236


In [6]:
loo = LeaveOneOut()

sample_size = min(300, len(X))

loo_scores = cross_val_score(
    model,
    X.iloc[:sample_size],
    y.iloc[:sample_size],
    cv=loo,
    scoring='neg_mean_squared_error'
)

print("LOOCV Average MSE")
print(-loo_scores.mean())


LOOCV Average MSE
6987180556663.917


In [7]:
tscv = TimeSeriesSplit(n_splits=5)

ts_scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring='r2'
)

print("Time Series Scores")
print(ts_scores)
print("Mean:", ts_scores.mean())


Time Series Scores
[0.90969325 0.91950341 0.91967411 0.91313971 0.92250898]
Mean: 0.9169038922158196


In [8]:
comparison = pd.DataFrame({
    'Method':['KFold','StratifiedKFold','TimeSeriesSplit'],
    'Mean_R2':[
        kfold_scores.mean(),
        strat_scores.mean(),
        ts_scores.mean()
    ],
    'Std_R2':[
        kfold_scores.std(),
        strat_scores.std(),
        ts_scores.std()
    ]
})

comparison.sort_values('Mean_R2', ascending=False)


,Method,Mean_R2,Std_R2
1,StratifiedKFold,0.917742,0.004223
0,KFold,0.917279,0.005978
2,TimeSeriesSplit,0.916904,0.004733



## Conclusion

- K-Fold provides robust evaluation.
- Stratified K-Fold preserves target distribution.
- LOOCV uses maximum training samples but is computationally expensive.
- TimeSeriesSplit is recommended when time order matters.
- Compare Mean R² and Std R² to choose the most stable strategy.
